In [14]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/big-data-final"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Instalar Java 17
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

# Instalar PySpark y driver
!pip install pyspark==3.5.0 cassandra-driver -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.5.0 which is incompatible.
Dependencias instaladas.


In [15]:
# Rutas del Data Lake
LANDING_PATH = f"{PROJECT_ROOT}/datalake/landing"
BRONZE_PATH = f"{PROJECT_ROOT}/datalake/bronze"
SILVER_PATH = f"{PROJECT_ROOT}/datalake/silver"
GOLD_PATH = f"{PROJECT_ROOT}/datalake/gold"
CHECKPOINT_PATH = f"{PROJECT_ROOT}/datalake/checkpoints"
QUARANTINE_PATH = f"{PROJECT_ROOT}/datalake/quarantine"

# Crear estructura de directorios si no existe
os.makedirs(LANDING_PATH, exist_ok=True)
os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(GOLD_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(QUARANTINE_PATH, exist_ok=True)

print(f"Directorios configurados en: {PROJECT_ROOT}")

Directorios configurados en: /content/drive/MyDrive/big-data-final


In [34]:
import shutil
from google.colab import userdata

# Credenciales de AstraDB
ASTRA_CLIENT_ID = userdata.get('ASTRA_CLIENT_ID')
ASTRA_CLIENT_SECRET = userdata.get('ASTRA_CLIENT_SECRET')

# Ruta al Secure Connect Bundle (SCB)
SCB_PATH = f"{PROJECT_ROOT}/secure-connect-cloud-analytics.zip"

if os.path.exists(SCB_PATH):
    print(f"Secure Connect Bundle encontrado en: {SCB_PATH}")
    abs_path = os.path.abspath(SCB_PATH)
    SCB_URI = f"file://{abs_path}"
else:
    print(f"No se encuentra el Secure Connect Bundle en {SCB_PATH}")
    raise FileNotFoundError(f"Sube el secure-connect-bundle.zip a {PROJECT_ROOT}")

if 'spark' in locals():
    spark.stop()

Secure Connect Bundle encontrado en: /content/drive/MyDrive/big-data-final/secure-connect-cloud-analytics.zip


In [35]:
# SparkSession
# Configuramos Spark para que descargue automáticamente el conector de Cassandra y utilice el SCB.
from pyspark.sql import SparkSession

SPARK_PACKAGES = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"

spark = SparkSession.builder \
    .appName("CloudProviderAnalytics_Colab") \
    .master("local[*]") \
    .config("spark.jars.packages", SPARK_PACKAGES) \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .config("spark.sql.catalog.myCatalog", "com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.cassandra.connection.config.cloud.path", SCB_URI) \
    .config("spark.cassandra.auth.username", ASTRA_CLIENT_ID) \
    .config("spark.cassandra.auth.password", ASTRA_CLIENT_SECRET) \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark Session iniciada. Versión: {spark.version}")

Spark Session iniciada. Versión: 3.5.0


In [44]:
# Smoke test para verificar la conexión

# 1. Test Spark Local
try:
    print("Test 1: Spark Local DataFrame...")
    spark.range(3).show()
    print("✅ Spark local OK.")
except Exception as e:
    print(f"❌ Error Spark local: {e}")

# 2. Test Conectividad AstraDB
print("\nTest 2: Conexión a AstraDB...")
try:
    # Usamos el catálogo 'myCatalog' que definimos en la configuración
    # Esto le pide a AstraDB la lista de Keyspaces visibles
    spark.sql("SHOW NAMESPACES IN myCatalog").show()
    print("✅ Verifica la existencia de 'cloud_analytics'")
    spark.sql("DESCRIBE NAMESPACE myCatalog.cloud_analytics").show(truncate=False)

except Exception as e:
    print(f"❌ Error al listar bases de datos: {e}")

Test 1: Spark Local DataFrame...
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+

✅ Spark local OK.

Test 2: Conexión a AstraDB...
+------------------+
|         namespace|
+------------------+
|   cloud_analytics|
|data_endpoint_auth|
|      datastax_sla|
+------------------+

✅ Verifica la existencia de 'cloud_analytics'
+--------------+---------------+
|info_name     |info_value     |
+--------------+---------------+
|Catalog Name  |myCatalog      |
|Namespace Name|cloud_analytics|
+--------------+---------------+

